(function-maps)=
# Function Map Application

The {ref}`data tree structures <data-tree>` allow for the application of specific functions on each of their nodes.
While it is easy to apply a single function on all nodes of a data tree, CRACE provides facilities to "map" different functions to subtrees or individual nodes.
At the same time, CRACE also supports "storing" functions in a dedicated registry, making it possible to identify them with their name, too.

Nodes can be identified via their path or their name (the last segment of their path).
Functions can be specified directly as callable objects which transform an {py:class}`xarray.Dataset` into another one.
The general signature is
```python
def func(arg: xarray.Dataset) -> xarray.Dataset: ...
```
Xarray objects generally support {external+numpy:ref}`universal functions<ufuncs-basics>`, which means that many functions that utilize universal functions with the signature
```python
def func(arg: numpy.ArrayLike) -> numpy.ArrayLike: ...
```
will also work.

Another way of specifying functions is by their name.
This is possible when working with a function registry, which by default is enabled for impact functions.
Impact functions may be registered in this registry with the {py:func}`~crace.impact_function` decorator.

In [ ]:
import xarray as xr
import numpy as np
import crace as ce

In [ ]:
dset = xr.Dataset({"var": (["x", "y"], np.zeros((3, 2)))})
tree = xr.DataTree.from_dict(
    {
        "/": None,
        "/a": dset.copy(),
        "/a/1": dset.copy(),
        "/a/2": dset.copy(),
        "/b": dset.copy(),
        "/b/1": dset.copy(),
        "/b/1/x": dset.copy(),
        "/b/2": dset.copy(),
        "/b/3": None,
        "/c": dset.copy(),
    }
)
tree

## Mapping Impact Functions

Inside {py:class}`crace.Engine`, impact functions are applied to the {py:attr}`~crace.Engine.hazard` data tree (which is isomorphic to {py:attr}`~crace.Engine.exposure`) to yield the pixel- or point-wise ratio of impacted exposure.
The single parameter of the function is therefore the hazard intensity (or other hazard measures if the dataset contains multiple data variables).
The result of the function is called the "impact ratio" and is then multiplied with {py:attr}`~crace.Engine.exposure` to yield the impact.

For the purpose of CRACE, "impact functions" are functions that transform data variables of datasets while keeping all dimensions and coordinates intact and follow the above signatures.
The distinction to other functions is only chosen in function and parameter names, as applying an "impact function" (as opposed to other types of functions) might have specific implications, as shown further below.

{py:class}`~crace.Engine` performs this application via {py:func}`~crace.map_impact_function`, which will usually not be called in user code.
However, the application of the impact functions in {py:class}`~crace.Engine` thus follow the exact rules of this function, which is why we investigate it here.

We define a function map as dictionary with keys of type {py:class}`str` or {py:class}`crace.FuncType`, and values of type {py:class}`~collections.abc.Callable` or {py:class}`str`.

In [ ]:
@ce.impact_function(name="my_func")
def plus_zero(x: xr.Dataset) -> xr.Dataset:
    return x


def plus_three(x: xr.Dataset) -> xr.Dataset:
    return x + 3


func_map = {
    "a": lambda x: x + 1,
    "1": lambda x: x + 2,
    "/b/2": plus_three,
    ce.FuncType.leaf: "my_func",  # Same as 'plus_zero'
}
ce.map_impact_function(tree, func_map)

Following the rules set by {py:func}`~crace.map_impact_function`, the tree nodes were matched with the following keys:

```{list-table}
:widths: "auto"
:header-rows: 1

*   - `tree` Node
    - `func_map` Key
    - Matching Rule
    - Result
*   - `"/"`
    - *no match*
    - 
    - No dataset (no match)
*   - `"/a"`
    - `"a"`
    - Matched by name
    - 1
*   - `"/a/1"`
    - `"1"`
    - Matched by name
    - 2
*   - `"/a/2"`
    - `"a"`
    - Matched by parent
    - 1
*   - `"/b"`
    - *no match*
    - 
    - No dataset (no match)
*   - `"/b/1"`
    - `"1"`
    - Matched by name
    - 2
*   - `"/b/1/x"`
    - `"1"`
    - Matched by parent
    - 2
*   - `"/b/2"`
    - `"/b/2"`
    - Matched by path
    - 3
*   - `"/b/3"`
    - `ce.FuncType.leaf`
    - Matched via leaf property
    - No dataset (no original)
*   - `"/c"`
    - `ce.FuncType.leaf`
    - Matched via leaf property
    - 0
```

## Mapping Aggregate Functions

Aggregate functions follow the same signature(s) as above, but they reduce over at least one of the dimensions in the input data.
Therefore, they will a return a dataset that has a different shape than the input dataset.

The application of "aggregate functions" differs slightly from that of "impact functions".
When applying an impact function to a data tree, we want to ensure that it is easy to specify how *all* nodes are transformed.
Obiously, each point of a hazard dataset should be transformed by an impact function.
This is different from aggregates, as these might only be required at specific levels or nodes of the tree.
For example, one might want to apply varying impact functions for different regions within a country, but only compute the total or average impact for the entire country.

The application of the function mapping algorithm thus only changes slightly, and this is handled by calling {py:func}`~crace.map_aggregate_function`.

In [ ]:
dset2 = xr.Dataset(
    {"var": (["x", "y"], np.ones((4, 2)))},
    coords={"x": np.arange(4), "y": np.arange(2)},
)
tree2 = xr.DataTree.from_dict(
    {
        "/": None,
        "/a": dset2.copy().sel(x=slice(0, 1)),
        "/b": dset2.copy().sel(x=slice(2, 3)),
    }
)
tree2

In [ ]:
agg_map = {
    "/": lambda x: x.sum(dim="x"),
    "/a": lambda x: x.sum(dim="y"),
    "/b": lambda x: x.sum(dim=["x", "y"]),
}
ce.map_aggregate_function(tree2, agg_map)